# Khmer Sentence Simplification v6: Complex → Simple (fewer wrong words)

This notebook uses **4 models**:
- **mT5-small seed 42**: learning rate 5e-4, no label smoothing.
- **mT5-small seed 43** and **mT5-small seed 44**: learning rate 3e-4, label smoothing 0.1.
- **XLM-RoBERTa-base**: pretrained, not trained here. It only scores how fluent a candidate sentence is.

The three mT5-small models rewrite the sentences. XLM-RoBERTa only scores them.

1. **Khmer fluency check.** XLM-RoBERTa scores every candidate sentence by pseudo-log-likelihood. It was trained on CC-100 web text, which includes Khmer. A garbled word such as "យវតា" usually scores much lower than the correct word, so the garbled candidate tends to lose.
2. **Three-model vote.** Seeds 42, 43 and 44 each generate 5 candidates. A candidate that agrees with the other models' first choices gets a bonus.
3. **Output clean-up.**
  - Royal words are replaced with everyday words, for example ព្រះចិន្ដា→ចិត្ត, ព្រះសព→សព, ព្រះទម្រង់→រូបរាង, ព្រះបាទ→ស្ដេច, ទូលព្រះបង្គំ→ខ្ញុំ. Each rule was tested on the training and validation data and kept only if it did not lower similarity.
  - Repeated words are removed.
4. **Honest selection.** Eight settings are compared on the validation set: seed 42 alone or the three-model vote, each with fluency weight 0, 10, 20 or 40. The baseline is seed 42 alone with weight 0 plus clean-up. A setting is allowed only if its chrF/SARI score is no more than 0.5 below that baseline. Among the allowed settings, the most fluent one is used. The test set is scored only once.

Note: with only one reference sentence, chrF/SARI barely penalize a misspelled or garbled word, which is why fluency breaks the tie. The real proof is your team's rating.


## 1. Setup

In [1]:
!pip -q install khmer-nltk transformers datasets sentencepiece sacrebleu accelerate

import os, re, gc, time, random, collections, math
import numpy as np, pandas as pd, torch, sacrebleu
from google.colab import files
IN_COLAB = True          # used later to save the models to Google Drive

# Fix the random seed so results can be repeated
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Check for a GPU
if not torch.cuda.is_available():
    raise RuntimeError('No GPU! In Colab')
print('GPU OK:', torch.cuda.get_device_name(0))

GPU OK: Tesla T4


## 1b. Save the trained model to Google Drive
If Colab disconnects, re-running the training cell then loads the saved model in seconds instead of training again. If Drive cannot be mounted, the notebook falls back to the temporary Colab disk.

In [2]:
USE_DRIVE = IN_COLAB          # Google Drive only exists in Colab
SAVE_ROOT = 'saved_models'
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_ROOT = '/content/drive/MyDrive/khmer_simplifier_models_v4'
    except Exception as e:
        print('Drive not available, saving on the temporary disk instead:', e)
os.makedirs(SAVE_ROOT, exist_ok=True)
print('models are saved in:', SAVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
models are saved in: /content/drive/MyDrive/khmer_simplifier_models_v4


## 2. Load the parallel pairs
Upload dataset (columns: source, complex, simple).

In [3]:
# Load the dataset (upload it if it is not in the Colab folder yet)
FN = 'complex_to_simple_pairs_1000.csv'
if not os.path.exists(FN):
    up = files.upload(); FN = list(up.keys())[0]

# Clean: drop empty rows, trim spaces, remove duplicate sentences
df = pd.read_csv(FN).dropna(subset=['complex', 'simple'])
df['complex'] = df.complex.str.strip()
df['simple'] = df.simple.str.strip()
df = df.drop_duplicates('complex').reset_index(drop=True)

# Quick summary and two example pairs
print(len(df), 'pairs from', df.source.nunique(), 'stories')
print('mean chars  complex:', round(df.complex.str.len().mean()), ' simple:', round(df.simple.str.len().mean()))
for i in range(2):
    print('\nCOMPLEX:', df.complex[i])
    print('SIMPLE :', df.simple[i])

1030 pairs from 115 stories
mean chars  complex: 87  simple: 71

COMPLEX: ស្ដេចយកសក់មួយសរសៃ ទៅដាក់ក្រោមកម្រាលពូកនោះ ដល់ពេលយប់ ទ្រង់ហៅបុរសនោះឲ្យទៅដេក ។
SIMPLE : ស្ដេចយកសក់មួយសរសៃទៅដាក់ក្រោមពូក ។ ដល់យប់ ស្ដេចហៅបុរសនោះឲ្យទៅដេក ។

COMPLEX: ព្រះរាជកុមារ ថ្វាយបង្គំលាព្រះមាតារួចស្រេច ក៏ម្នីម្នាតម្រង់ទៅ ដែនពារាណសី ដែលជាទីឃុំឃាំងព្រះបិតាព្រះអង្គ ។
SIMPLE : ព្រះរាជកុមារលាម្ដាយរួច ក៏ប្រញាប់ទៅដែនពារាណសី ជាកន្លែងដែលគេឃុំឪពុករបស់គាត់ ។


## 3. Train / validation / test split (by story)
Validation and test sentences come from stories the model never saw. Validation picks the decoding method and the best epoch; test is scored once at the end.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

# Split by story, so no story appears in two sets
def make_split(df, seed=42):
    # Step 1: take 10% of the stories as the test set
    split1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=seed)
    rest_idx, test_idx = next(split1.split(df, groups=df.source))
    rest = df.iloc[rest_idx].reset_index(drop=True)
    test = df.iloc[test_idx].reset_index(drop=True)

    # Step 2: from the remaining stories, take 1/9 as validation (about 10% of all data)
    split2 = GroupShuffleSplit(n_splits=1, test_size=1/9, random_state=seed)
    train_idx, val_idx = next(split2.split(rest, groups=rest.source))
    train = rest.iloc[train_idx].reset_index(drop=True)
    val = rest.iloc[val_idx].reset_index(drop=True)
    return train, val, test

train, val, test = make_split(df, SEED)
print('train', len(train), '| val', len(val), '| test', len(test))

# Check: the number of shared stories should be 0 for every pair of sets
overlap = lambda a, b: len(set(a.source) & set(b.source))
print('story overlap  train/val:', overlap(train, val),
      '| train/test:', overlap(test, train), '| val/test:', overlap(val, test))

train 821 | val 110 | test 99
story overlap  train/val: 0 | train/test: 0 | val/test: 0


## 4. Data consistency check
Flags pairs that may hurt training (identical, much longer, or very different).

In [5]:
# Data check: find pairs that might hurt training
d = df.copy()
d['len_ratio'] = d.simple.str.len() / d.complex.str.len()      # simple length ÷ complex length
d['similarity'] = [sacrebleu.sentence_chrf(s, [c]).score       # how similar the two sentences are (chrF)
                   for c, s in zip(d.complex, d.simple)]

# Four warning signs
flags = {
    'identical to input':           d[d.simple == d.complex],   # nothing was simplified
    'rewrite >15% longer':          d[d.len_ratio > 1.15],      # simple version is longer
    'rewrite <50% of input length': d[d.len_ratio < 0.5],       # too much was removed
    'very different (chrF<25)':     d[d.similarity < 25],       # meaning may have changed
}
for name, rows in flags.items():
    print(f'{name:<32}: {len(rows)}')

# Save every flagged pair once, so it can be reviewed by hand
bad = pd.concat(flags.values()).drop_duplicates('complex')
bad[['source', 'complex', 'simple']].to_csv('pairs_to_review.csv', index=False, encoding='utf-8-sig')
print('saved pairs_to_review.csv with', len(bad), 'pairs')

identical to input              : 0
rewrite >15% longer             : 0
rewrite <50% of input length    : 4
very different (chrF<25)        : 31
saved pairs_to_review.csv with 31 pairs


## 5. Royal-word glossary (pre-pass)
A list of royal / literary words with their everyday equivalents. It is applied **before** the model, to the training, validation and test inputs alike. The glossary was written from general knowledge of Khmer royal vocabulary and each rule was checked against the training data (rules that made the rewrite *less* similar to the reference were removed).

Longest match wins, so ព្រះរាជបុត្រី→កូនស្រី is applied before ព្រះរាជបុត្រ→កូន. Words in `PROTECT` are never changed.

In [6]:
# ===== 1. Glossary: royal word -> everyday word (applied to the model INPUT) =====
GLOSSARY = {
    # body, food and feelings
    'ព្រះខ័ន': 'ដាវ',
    'ព្រះកាយ': 'ខ្លួន',
    'ព្រះភក្ត្រ': 'មុខ',
    'ព្រះហស្ត': 'ដៃ',
    'ព្រះនេត្រ': 'ភ្នែក',
    'ព្រះឱស្ឋ': 'មាត់',
    'ព្រះកេស': 'សក់',
    'ព្រះពាហា': 'ដៃ',
    'ព្រះសូរសៀង': 'សំឡេង',
    'ព្រះសុរសៀង': 'សំឡេង',
    'ព្រះភោជន': 'បាយ',
    'ភោជនាហារ': 'អាហារ',
    'ព្រះទ័យ': 'ចិត្ត',
    'ព្រះរាជហឫទ័យ': 'ចិត្ត',
    'សព្វព្រះរាជហឫទ័យ': 'ពេញចិត្ត',
    # places
    'ព្រះរាជវាំង': 'វាំង',
    'ព្រះបរមរាជវាំង': 'វាំង',
    'ព្រះរាជដំណាក់': 'ដំណាក់',
    'ព្រះរាជពិធី': 'ពិធី',
    'ព្រះនគរ': 'នគរ',
    'ព្រះរាជាណាចក្រ': 'នគរ',
    # family
    'ព្រះរាជបុត្រី': 'កូនស្រី',
    'ព្រះរាជបុត្រា': 'កូនប្រុស',
    'ព្រះរាជបុត្រ': 'កូន',
    'ព្រះរាជធីតា': 'កូនស្រី',
    'ព្រះបិតា': 'ឪពុក',
    'ព្រះមាតា': 'ម្ដាយ',
    'ព្រះជនក': 'ឪពុក',
    'ព្រះជននី': 'ម្ដាយ',
    'ព្រះស្វាមី': 'ប្ដី',
    'ព្រះអនុជ': 'ប្អូន',
    'ព្រះស្នំ': 'ស្នំ',
    'ព្រះនាង': 'នាង',
    'ព្រះរាជវង្សានុវង្ស': 'ញាតិវង្ស',
    'ព្រះញាតិវង្ស': 'ញាតិវង្ស',
    # king
    'ព្រះមហាក្សត្រ': 'ស្ដេច',
    'ព្រះរាជាធិរាជ': 'ស្ដេច',
    'ព្រះរាជា': 'ស្ដេច',
    'ព្រះចៅ': 'ស្ដេច',
    # royal verbs
    'ទតឃើញ': 'ឃើញ',
    'ត្រាស់បង្គាប់': 'បង្គាប់',
    'ត្រាស់សួរ': 'សួរ',
    'ត្រាស់ហៅ': 'ហៅ',
    'ត្រាស់': '',
    'យាងទៅ': 'ទៅ',
    'យាងមក': 'មក',
    'សោយរាជ្យ': 'គ្រងរាជ្យ',
    'ថ្វាយបង្គំលា': 'លា',
    'ក្រាបទូល': 'ប្រាប់',
    'ក្រាបបង្គំទូល': 'ប្រាប់',
    'ទ្រង់ព្រះសណ្ដាប់': 'ស្ដាប់',
    'ទិវង្គត': 'ស្លាប់',
    'ព្រះរាជសារ': 'សំបុត្រ',
    # royal words that are simply removed
    'ទ្រង់': '',
    'ព្រះអង្គ': '',
    'ព្រះរាជឱង្ការ': '',
}
PROTECT = ['ព្រះអង្គម្ចាស់', 'ត្រាស់ដឹង']      # never changed (a royal title and "to become enlightened")

def make_replacer(rules):
    # replaces every rule in a sentence, longest word first; protected words stay as they are
    rules = {**rules, **{p: p for p in PROTECT}}
    pattern = re.compile('|'.join(re.escape(k) for k in sorted(rules, key=len, reverse=True)))
    return lambda s: pattern.sub(lambda m: rules[m.group(0)], s)

replace_input = make_replacer(GLOSSARY)
def normalize(s):
    return re.sub(r'\s+', ' ', replace_input(s)).strip()

# show 3 examples, then apply the glossary to every input
for s in train.complex.head(3):
    print('RAW :', s)
    print('NORM:', normalize(s))
    print()
for part in (train, val, test):
    part['x'] = part.complex.apply(normalize)       # x = the model input

# ===== 2. Clean-up of the model OUTPUT =====
# 19 extra royal words; each was kept only if it made train+val outputs MORE similar to the reference
POST_GLOSS = {
    'ព្រះចិន្ដា': 'ចិត្ត', 'ព្រះសព': 'សព', 'ព្រះទម្រង់': 'រូបរាង', 'ព្រះតម្រិះ': 'គំនិត', 'ព្រះជន្ម': 'អាយុ', 'ព្រះរោគ': 'ជំងឺ',
    'សោយរាជសម្បត្តិ': 'គ្រងរាជ្យ', 'រាជសម្បត្តិ': 'រាជ្យ', 'ព្រះដំណើរ': 'ដំណើរ', 'ព្រះគុណ': 'គុណ', 'ព្រះមេត្តា': 'មេត្តា',
    'មានព្រះបន្ទូលថា': 'ប្រាប់ថា', 'មានព្រះបន្ទូល': 'ប្រាប់', 'ព្រះបាទ': 'ស្ដេច', 'ព្រះទីន័ង': 'ទីន័ង', 'ព្រះលាន': 'លាន',
    'ទូលព្រះបង្គំ': 'ខ្ញុំ', 'ក្រាបបង្គំលា': 'លា', 'ក្រាបបង្គំ': 'ក្រាប',
}
replace_output = make_replacer({**GLOSSARY, **POST_GLOSS})

# finds a word repeated right after itself, e.g. "ស្ដេចស្ដេច" -> "ស្ដេច"
_words = sorted({v for v in [*GLOSSARY.values(), *POST_GLOSS.values()] if len(v) >= 3}, key=len, reverse=True)
REPEAT_RE = re.compile('(' + '|'.join(re.escape(w) for w in _words) + r')(?:\s*\1)+')

def post(s):
    s = replace_output(s)          # leftover royal words -> everyday words
    s = REPEAT_RE.sub(r'\1', s)    # remove repeated words
    return re.sub(r'\s+', ' ', s).strip()

print('post-edit self-test:', post('ស្ដេចស្ដេចបង្គាប់ព្រះចិន្ដាព្រះសព ។'))
print('rules:', len(GLOSSARY), '| inputs changed by the glossary (test):', round((test.x != test.complex).mean(), 2))

RAW : ព្រះរាជកុមារ ថ្វាយបង្គំលាព្រះមាតារួចស្រេច ក៏ម្នីម្នាតម្រង់ទៅ ដែនពារាណសី ដែលជាទីឃុំឃាំងព្រះបិតាព្រះអង្គ ។
NORM: ព្រះរាជកុមារ លាម្ដាយរួចស្រេច ក៏ម្នីម្នាតម្រង់ទៅ ដែនពារាណសី ដែលជាទីឃុំឃាំងឪពុក ។

RAW : ព្រះរាជបុត្រីនេះ មានរូបឆោមលោមពណ៌ស្រស់ឆើតឆាយណាស់ រកនរណាឲ្យល្អត្រឹមស្មើពុំបានឡើយ ។
NORM: កូនស្រីនេះ មានរូបឆោមលោមពណ៌ស្រស់ឆើតឆាយណាស់ រកនរណាឲ្យល្អត្រឹមស្មើពុំបានឡើយ ។

RAW : ស្ដេចទ្រង់ជ្រាប ទ្រង់ផ្សំនាងពៅឲ្យជាជាយាចៅខ្វាក់ ឲ្យសោយរាជ្យជំនួសព្រះអង្គ ។
NORM: ស្ដេចជ្រាប ផ្សំនាងពៅឲ្យជាជាយាចៅខ្វាក់ ឲ្យគ្រងរាជ្យជំនួស ។

post-edit self-test: ស្ដេចបង្គាប់ចិត្តសព ។
rules: 56 | inputs changed by the glossary (test): 0.55


## 6. Evaluation helpers
- **SARI-1**: unigram keep / add / delete quality of the edits (higher = better).
- **Copy rate**: outputs identical to the input (lower = really rewrites).
- **Formal left**: outputs still containing royal words (the reference value is shown for calibration).
- **Too short**: outputs under 60% of the input length (possible dropped content).
- **Missing words**: average number of *content words of the input that the training data almost always keeps* but that are absent from the output. A high value means dropped or replaced content (this is how a wrong word like sword→"mother" shows up). The tokenizer is imperfect, so the reference also scores above 0: compare the models with each other and with the reference value.

In [7]:
from khmernltk import word_tokenize

# ---------- Khmer word helpers ----------
tok = lambda s: [w for w in word_tokenize(s) if w.strip()]        # split Khmer text into words (no spaces in Khmer)
ISKH = lambda w: bool(re.fullmatch(r'[ក-ឳា-៑]{2,}', w))           # True if w is a real Khmer word (2+ Khmer letters)

# ---------- Royal-word detector ----------
FORMAL_WORDS = set(GLOSSARY) | set(POST_GLOSS) | {'ថ្វាយបង្គំ', 'ក្រាបទូល'}
FORMAL_RE = re.compile('|'.join(re.escape(w) for w in sorted(FORMAL_WORDS, key=len, reverse=True)))
FORMAL_TOKENS = set()          # filled after the lexicon is learned (train data only)

def has_formal(s):
    # True if the sentence still contains a royal word (protected titles are ignored)
    s_check = s
    for p in PROTECT:
        s_check = s_check.replace(p, '')
    return bool(FORMAL_RE.search(s_check)) or any(w in FORMAL_TOKENS for w in tok(s))

# ---------- SARI-1: did the model keep, add and delete the right words? ----------
def f1(chosen, correct):
    # F1 score between two word sets (1.0 when both are empty)
    if not chosen and not correct:
        return 1.0
    hits = len(chosen & correct)
    precision = hits / len(chosen) if chosen else 0.0
    recall = hits / len(correct) if correct else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def sari1(src, pred, ref):
    S, P, R = set(tok(src)), set(tok(pred)), set(tok(ref))    # words of source, prediction, reference
    keep = f1(S & P, S & R)                                   # kept the right words?
    add = f1(P - S, R - S)                                    # added the right new words?
    deleted, should_delete = S - P, S - R
    if not deleted and not should_delete:
        delete = 1.0
    else:
        delete = len(deleted & should_delete) / len(deleted) if deleted else 0.0   # deleted the right words?
    return 100 * (keep + add + delete) / 3

# ---------- All metrics for one set of outputs ----------
MISSING = None   # defined after the baselines (needs train statistics)

def evaluate(preds, refs, srcs, name, xs=None, verbose=True):
    xs = xs or srcs
    avg = lambda values: float(np.mean(values))
    pairs = list(zip(preds, srcs))
    m = dict(
        model=name,
        chrF=sacrebleu.corpus_chrf(preds, [refs]).score,                                  # character overlap with reference
        BLEU=sacrebleu.corpus_bleu(preds, [refs], tokenize='char').score,                 # character BLEU
        SARI1=avg([sari1(s, p, r) for s, p, r in zip(srcs, preds, refs)]),                # right edits
        copy_rate=avg([p.strip() == s.strip() for p, s in pairs]),                        # output = input
        formal_left=avg([has_formal(p) for p in preds]),                                  # royal words left
        too_short=avg([len(p) < 0.6 * len(s) for p, s in pairs]),                         # under 60% of input length
        missing=avg([len(MISSING(p, x)) for p, x in zip(preds, xs)]) if MISSING else float('nan'),  # key words lost
        len_ratio=avg([len(p) / max(1, len(s)) for p, s in pairs]),                       # output length / input length
    )
    if verbose:
        print(f"{name:<26} chrF {m['chrF']:5.1f} | SARI1 {m['SARI1']:5.1f} | copy {m['copy_rate']:.2f} | "
              f"formal {m['formal_left']:.2f} | short {m['too_short']:.2f} | missing {m['missing']:.2f} | len {m['len_ratio']:.2f}")
    return m

## 7. Baselines
**Copy input**, **glossary only**, and **glossary + lexicon** (word replacements learned from the training pairs: a word that usually disappears is mapped to the word that most often appears instead).

In [8]:
# ===== 1. Learn a word-replacement lexicon from the training pairs =====
train_in = [tok(s) for s in train.x]          # words of each complex sentence (after the glossary)
train_out = [tok(s) for s in train.simple]    # words of each simple sentence

seen = collections.Counter(w for words in train_in for w in set(words))   # number of sentences containing each word
gone = collections.Counter()                                               # how often the word disappears in the simple version
replaced_by = collections.defaultdict(collections.Counter)                 # which new words appear when it disappears
for c_words, s_words in zip(train_in, train_out):
    c_set, s_set = set(c_words), set(s_words)
    new_words, dropped = s_set - c_set, c_set - s_set
    for w in dropped:
        gone[w] += 1
        for n in new_words:
            replaced_by[w][n] += 1

lex = {}
for w, n in seen.items():
    # a Khmer word seen in 3+ sentences that is dropped in at least 80% of the rewrites
    if n >= 3 and gone[w] / n >= 0.8 and ISKH(w):
        options = [(new, count) for new, count in replaced_by[w].most_common(5) if ISKH(new)]
        if options and options[0][1] / gone[w] >= 0.4:
            lex[w] = options[0][0]        # replace it with its most common substitute
        elif gone[w] / n >= 0.95:
            lex[w] = ''                   # almost always deleted -> delete it
print(len(lex), 'lexicon rules learned from train')
FORMAL_TOKENS.update(w for w in lex if len(w) >= 3)   # learned words also count as royal/formal

def lexicon_simplify(x):
    return ''.join(lex.get(w, w) for w in word_tokenize(x))

# ===== 2. "Missing words": important input words that vanished from an output =====
FUNC = set(w for w, _ in collections.Counter(w for words in train_out for w in words).most_common(60))   # 60 most common words

def must_keep(w):
    # an input word the training rewrites (almost) always keep; unseen words count as must-keep
    if not ISKH(w) or w in FUNC or w in lex:
        return False
    if seen.get(w, 0) == 0:
        return True
    return gone[w] / seen[w] < 0.5

def missing_words(cand, x):
    return [w for w in tok(x) if must_keep(w) and w not in cand]
MISSING = missing_words

# ===== 3. Baselines on the validation set =====
print('\n--- VALIDATION ---')
vx, vref, vsrc = list(val.x), list(val.simple), list(val.complex)
evaluate(vsrc, vref, vsrc, 'copy input', vx)                                      # no change at all
evaluate(vx, vref, vsrc, 'glossary only', vx)                                     # glossary pass only
evaluate([lexicon_simplify(x) for x in vx], vref, vsrc, 'glossary + lexicon', vx) # glossary + learned word list
evaluate(vref, vref, vsrc, 'reference (calibration)', vx)                         # the human answer itself

| 2026-11-25 10:00:25,777 | INFO | khmer-nltk | Loaded model from /usr/local/lib/python3.13/dist-packages/khmernltk/word_tokenize/sklearn_crf_ner_10000.sav |
INFO:khmer-nltk:Loaded model from /usr/local/lib/python3.13/dist-packages/khmernltk/word_tokenize/sklearn_crf_ner_10000.sav


61 lexicon rules learned from train

--- VALIDATION ---
copy input                 chrF  67.5 | SARI1  28.6 | copy 1.00 | formal 0.79 | short 0.00 | missing 0.02 | len 1.00
glossary only              chrF  69.4 | SARI1  44.9 | copy 0.57 | formal 0.57 | short 0.01 | missing 0.00 | len 0.95
glossary + lexicon         chrF  70.8 | SARI1  60.8 | copy 0.20 | formal 0.10 | short 0.01 | missing 0.00 | len 0.93
reference (calibration)    chrF 100.0 | SARI1 100.0 | copy 0.00 | formal 0.04 | short 0.01 | missing 0.85 | len 0.83


{'model': 'reference (calibration)',
 'chrF': 100.0,
 'BLEU': 100.00000000000004,
 'SARI1': 100.0,
 'copy_rate': 0.0,
 'formal_left': 0.03636363636363636,
 'too_short': 0.00909090909090909,
 'missing': 0.8454545454545455,
 'len_ratio': 0.8307862771988997}

## 8. Guarded decoding (how bad candidates are rejected)
The model returns 5 candidates (beam search). Each is scored: the model's own ranking is the base, and a candidate is **penalised** if it is too short (<65% of the input), if it loses content words of the input that are normally kept (this catches wrong-word rewrites such as sword → "mother"), or if it still contains royal words. The lexicon output is added as a last-resort candidate. The best-scoring candidate wins.

In [9]:
def cand_score(c, x, rank):
    if not c.strip(): return -1e9
    s = 0.25*sacrebleu.sentence_chrf(c, [x]).score - 3*rank          # prefer top-ranked, mildly prefer faithful
    s -= 8*len(missing_words(c, x))                                    # lost / replaced content words
    if len(c) < 0.65*len(x): s -= 40                                   # dropped content
    if has_formal(c): s -= 25                                          # royal words left
    return s

def pick(cands, x):
    pool = list(enumerate(cands)) + [(6, lexicon_simplify(x))]        # lexicon = safe last resort
    return max(pool, key=lambda p: cand_score(p[1], x, p[0]))[1]

# quick self-test on a real failure from the earlier run (sword was rewritten as "mother")
_x = normalize('ព្រះកេតុមាលាទ្រង់ទតឃើញដូច្នោះ ក៏ត្រាស់បង្គាប់ឲ្យគេរត់តាមស្រែកហៅព្រះពិស្ណុការ ទារយកព្រះខ័នមកថ្វាយព្រះអង្គវិញ ។')
_bad = 'ស្ដេចឃើញដូច្នោះ ក៏បង្គាប់ឲ្យគេរត់តាមស្រែកហៅម្ដាយមកថ្វាយស្ដេចវិញ ។'
_ok  = 'ស្ដេចឃើញដូច្នោះ ក៏បង្គាប់ឲ្យគេរត់តាមហៅព្រះពិស្ណុការ ទារយកដាវមកវិញ ។'
print('normalized input :', _x)
print('score bad / good :', round(cand_score(_bad, _x, 0), 1), '/', round(cand_score(_ok, _x, 1), 1), '(the good one should score higher)')

normalized input : ព្រះកេតុមាលាឃើញដូច្នោះ ក៏បង្គាប់ឲ្យគេរត់តាមស្រែកហៅព្រះពិស្ណុការ ទារយកដាវមកថ្វាយវិញ ។
score bad / good : -9.9 / 6.4 (the good one should score higher)


## 9. Neural model
`run_model()` fine-tunes the model (used for seed 42 with the default settings, and for seeds 43 and 44 with learning rate 3e-4 and label smoothing 0.1) on the **glossary-normalised** inputs with warm-up, early stopping and best-epoch selection (lowest validation loss). Pairs that contain royal words are shown twice. The trained model is saved, and re-running the cell loads it instead of training again.



In [10]:
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq, EarlyStoppingCallback, set_seed)
from datasets import Dataset

# ===== Settings (only mT5-small is used) =====
CONFIGS = {'mt5-small': dict(name='google/mt5-small', maxlen=128, lr=5e-4, bs=16, no_repeat=3)}
EPOCHS = 10
OVERSAMPLE_FORMAL = True      # show pairs with royal words twice during training
PREFIX = 'simplify: '         # task instruction added in front of every input

# ===== Generate: beam search, returns n_ret candidate sentences per input =====
def generate(model, tk, cfg, texts, beams=4, n_ret=1, bs=16):
    model.eval()
    outs = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i + bs]
        enc = tk([PREFIX + t for t in chunk], return_tensors='pt', padding=True, truncation=True,
                 max_length=cfg['maxlen']).to(model.device)
        with torch.no_grad():
            o = model.generate(**enc, max_new_tokens=cfg['maxlen'], num_beams=max(beams, n_ret),
                               num_return_sequences=n_ret, no_repeat_ngram_size=cfg['no_repeat'])
        dec = [d.strip() for d in tk.batch_decode(o, skip_special_tokens=True)]
        outs += [dec[j * n_ret:(j + 1) * n_ret] for j in range(len(chunk))]
    return outs

# ===== Load the saved model, or fine-tune mT5 if it does not exist yet =====
def run_model(key, tr, va, epochs=EPOCHS, seed=SEED, lr=None, ls=0.0):
    cfg = CONFIGS[key]
    set_seed(seed)
    save_dir = os.path.join(SAVE_ROOT, f'{key}_norm_s{seed}')      # e.g. .../mt5-small_norm_s42

    # 1. Already trained? Load it from Drive (no training)
    if os.path.exists(os.path.join(save_dir, 'config.json')):
        print(f'Found saved model {save_dir} -> loading it (no training).')
        return AutoModelForSeq2SeqLM.from_pretrained(save_dir).to('cuda'), AutoTokenizer.from_pretrained(save_dir), cfg

    # 2. Otherwise fine-tune mT5 on our pairs
    t0 = time.time()
    tk = AutoTokenizer.from_pretrained(cfg['name'])
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg['name'])
    tr_use = tr
    if OVERSAMPLE_FORMAL:   # add a second copy of pairs that contain royal words, then shuffle
        tr_use = pd.concat([tr, tr[tr.complex.apply(has_formal)]]).sample(frac=1, random_state=seed).reset_index(drop=True)

    def prep(b):   # text -> token ids: input = 'simplify: ' + complex sentence, label = simple sentence
        x = tk([PREFIX + t for t in b['x']], max_length=cfg['maxlen'], truncation=True)
        y = tk(text_target=b['simple'], max_length=cfg['maxlen'], truncation=True)
        x['labels'] = y['input_ids']
        return x
    cols = ['x', 'simple']
    dtr = Dataset.from_pandas(tr_use[cols]).map(prep, batched=True, remove_columns=cols)
    dva = Dataset.from_pandas(va[cols]).map(prep, batched=True, remove_columns=cols)

    warm = int(0.05 * math.ceil(len(dtr) / cfg['bs']) * epochs)     # warm-up = 5% of all training steps
    args = Seq2SeqTrainingArguments(
        output_dir=f'out_{key}', per_device_train_batch_size=cfg['bs'], per_device_eval_batch_size=16,
        learning_rate=lr or cfg['lr'], label_smoothing_factor=ls, num_train_epochs=epochs, weight_decay=0.01,
        warmup_steps=warm, lr_scheduler_type='linear',
        eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,   # keep the best epoch
        logging_steps=20, fp16=False, report_to='none', seed=seed)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                             data_collator=DataCollatorForSeq2Seq(tk, model=model),
                             callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])   # stop after 2 epochs with no improvement
    trainer.train()

    hist = [h['eval_loss'] for h in trainer.state.log_history if 'eval_loss' in h]
    print(f'{key}: trained {len(hist)} epochs in {(time.time() - t0) / 60:.1f} min, best val loss {min(hist):.3f}')
    model.save_pretrained(save_dir)
    tk.save_pretrained(save_dir)
    print('saved to', save_dir)
    return model, tk, cfg

In [ ]:
# Score each mT5 seed on its own (validation set only, so the test set stays unseen)
print('\n--- VALIDATION: each seed alone (fluency w=0) ---')
for i, sd in enumerate(SEEDS):
    outs = decode_pool([vcand[i]], vx, 0, 0)
    m = evaluate(outs, vref, vsrc, f'seed {sd} only', vx)
    print(f'   combined score {sc(m):.2f} | fluency {np.mean(fluency(outs)):.3f}')

In [12]:
# ===== 1. Load the three mT5-small models: seed 42, seed 43, seed 44 (from Drive, no training) =====
SEEDS = [42, 43, 44]          # seed 42: lr 5e-4, no label smoothing; seeds 43, 44: lr 3e-4, label smoothing 0.1
models = []
for sd in SEEDS:
    print('\n' + '=' * 20, 'mt5-small seed', sd, '=' * 20)
    if sd == 42:
        m, tk_, cfg_ = run_model('mt5-small', train, val, seed=sd)                     # seed 42: default settings
    else:
        m, tk_, cfg_ = run_model('mt5-small', train, val, seed=sd, lr=3e-4, ls=0.1)    # seeds 43, 44: label-smoothed
    models.append((m, tk_, cfg_))
    gc.collect(); torch.cuda.empty_cache()

# ===== 2. Fluency scorer: how natural does a Khmer sentence sound? (XLM-RoBERTa) =====
from transformers import AutoModelForMaskedLM
flu_tk = AutoTokenizer.from_pretrained('FacebookAI/xlm-roberta-base')
flu_model = AutoModelForMaskedLM.from_pretrained('FacebookAI/xlm-roberta-base').to('cuda').eval()
flu_cache = {}

@torch.no_grad()
def fluency(sents, bs=48):
    # pseudo-log-likelihood: hide each token in turn, see how well the model guesses it back,
    # and average the log-probabilities (higher = more natural Khmer)
    for sent in dict.fromkeys(sents):              # each unique sentence once
        if sent in flu_cache:
            continue
        ids = flu_tk(sent, return_tensors='pt', truncation=True, max_length=128)['input_ids'][0]
        n = len(ids) - 2                           # tokens without the start/end markers
        if n <= 0:
            flu_cache[sent] = -20.0
            continue
        total = 0.0
        for a in range(0, n, bs):                  # mask up to 48 positions at once (one copy per position)
            pos = torch.arange(a + 1, min(a + bs, n) + 1)
            batch = ids.unsqueeze(0).repeat(len(pos), 1)
            batch[torch.arange(len(pos)), pos] = flu_tk.mask_token_id
            logits = flu_model(input_ids=batch.to('cuda')).logits             # (copies, tokens, vocabulary)
            masked = logits[torch.arange(len(pos)), pos.to('cuda')].float()   # keep only the hidden positions
            logp = torch.log_softmax(masked, dim=-1)                          # probabilities over the vocabulary
            total += logp[torch.arange(len(pos)), ids[pos].to('cuda')].sum().item()
        flu_cache[sent] = total / n
    return [flu_cache[s] for s in sents]

# sanity check on a known seed-42 mistake: the correct word should score higher (prints True)
correct_s = 'រុក្ខទេវតាក៏ផ្លាស់រូបជាទេវតាវិញភ្លាមៗ ។'
garbled_s = 'រុក្ខទេវតាក៏ផ្លាស់រូបជាយវតាវិញភ្លាមៗ ។'
g, b = fluency([correct_s, garbled_s])
print(f'fluency sanity: correct {g:.2f}  vs garbled {b:.2f} -> prefers the correct word: {g > b}')

# ===== 3. Decoding: collect candidates and pick the best one =====
def cands_of(i, xs):
    # 5 cleaned-up candidates per sentence from model i (beam search with 8 beams)
    return [[post(c) for c in lst] for lst in generate(*models[i], xs, beams=8, n_ret=5)]

def decode_pool(per_model, xs, w_flu=0.0, w_cons=0.0):
    # per_model = for each model, the candidate lists of every sentence; returns the best candidate per sentence
    out = []
    for j, x in enumerate(xs):
        tops = [pm[j][0] for pm in per_model]                               # each model's first choice
        pool = [(r, c) for pm in per_model for r, c in enumerate(pm[j])]    # (rank, candidate) from every model
        pool.append((6, post(lexicon_simplify(x))))                         # safe backup from the word list
        flu = dict(zip([c for _, c in pool], fluency([c for _, c in pool]))) if w_flu else {}

        def score(p):
            r, c = p
            v = cand_score(c, x, r)                                                          # guard: key words, length, royal words
            if w_cons:
                v += w_cons * np.mean([sacrebleu.sentence_chrf(c, [t]).score for t in tops])  # agreement between models
            if flu:
                v += w_flu * flu[c]                                                          # natural Khmer
            return v
        out.append(max(pool, key=score)[1])
    return out

# ===== 4. Choose the decoding setting on the VALIDATION set =====
print('\n--- VALIDATION: model(s) x fluency weight ---')
vcand = [cands_of(i, vx) for i in range(len(models))]
sc = lambda m: (m['chrF'] + m['SARI1']) / 2 - 50 * m['too_short'] - 5 * m['missing']   # one combined score

options, flu_mean = {}, {}
for pool_name, idxs, wc in (('seed 42 only', [0], 0.0), ('3-model vote', list(range(len(models))), 0.15)):
    for w in [0, 10, 20, 40]:                                   # fluency weight
        nm = f'{pool_name} | fluency w={w}'
        outs = decode_pool([vcand[i] for i in idxs], vx, w, wc)
        options[nm] = (evaluate(outs, vref, vsrc, nm, vx), (idxs, w, wc))
        flu_mean[nm] = float(np.mean(fluency(outs)))           # average naturalness of the chosen outputs

base = 'seed 42 only | fluency w=0'     # baseline: seed 42 + clean-up, no fluency score
# chrF/SARI cannot see wrong words, so choose in two steps:
#  1) keep settings whose score is at most 0.5 below the base
#  2) among them, take the most natural Khmer (highest average fluency)
okset = [k for k in options if sc(options[k][0]) >= sc(options[base][0]) - 0.5]
best_name = max(okset, key=lambda k: (flu_mean[k], sc(options[k][0])))
print(pd.DataFrame({'score': {k: round(sc(options[k][0]), 2) for k in options},
                    'fluency': {k: round(flu_mean[k], 3) for k in options}}).to_string())
idxs, W_FLU, W_CONS = options[best_name][1]
print(f"\nSelected on validation: {best_name}   (score {sc(options[best_name][0]):.2f} vs seed 42 + clean-up {sc(options[base][0]):.2f}; "
      f"fluency {flu_mean[best_name]:.3f} vs {flu_mean[base]:.3f})")
best = dict(key='mt5-small', decode=best_name)
DECODE = lambda xs: decode_pool([cands_of(i, xs) for i in idxs], xs, W_FLU, W_CONS)


==================== mt5-small seed 42 ====================
Found saved model /content/drive/MyDrive/khmer_simplifier_models_v4/mt5-small_norm_s42 -> loading it (no training).


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



==================== mt5-small seed 43 ====================
Found saved model /content/drive/MyDrive/khmer_simplifier_models_v4/mt5-small_norm_s43 -> loading it (no training).


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


... (2 outputs left out)

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] XLMRobertaForMaskedLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


fluency sanity: correct -2.14  vs garbled -3.51 -> prefers the correct word: True

--- VALIDATION: model(s) x fluency weight ---
seed 42 only | fluency w=0 chrF  73.3 | SARI1  70.3 | copy 0.04 | formal 0.00 | short 0.01 | missing 0.09 | len 0.87
seed 42 only | fluency w=10 chrF  73.3 | SARI1  70.4 | copy 0.02 | formal 0.00 | short 0.01 | missing 0.08 | len 0.87
seed 42 only | fluency w=20 chrF  73.5 | SARI1  69.8 | copy 0.04 | formal 0.01 | short 0.02 | missing 0.09 | len 0.87
seed 42 only | fluency w=40 chrF  73.2 | SARI1  69.1 | copy 0.05 | formal 0.03 | short 0.02 | missing 0.08 | len 0.88
3-model vote | fluency w=0 chrF  73.8 | SARI1  70.3 | copy 0.03 | formal 0.00 | short 0.01 | missing 0.07 | len 0.88
3-model vote | fluency w=10 chrF  73.4 | SARI1  70.0 | copy 0.02 | formal 0.00 | short 0.02 | missing 0.06 | len 0.88
3-model vote | fluency w=20 chrF  73.2 | SARI1  69.7 | copy 0.02 | formal 0.00 | short 0.02 | missing 0.07 | len 0.88
3-model vote | fluency w=40 chrF  72.9 | SARI1 

## 10. Final test evaluation

In [13]:
tx, tref, tsrc = list(test.x), list(test.simple), list(test.complex)
final = DECODE(tx)
rows = [evaluate(tsrc, tref, tsrc, 'copy input', tx),
        evaluate(tx, tref, tsrc, 'glossary only', tx),
        evaluate([lexicon_simplify(x) for x in tx], tref, tsrc, 'glossary + lexicon', tx),
        evaluate(final, tref, tsrc, f"FINAL v6 ({best['decode']})", tx),
        evaluate(tref, tref, tsrc, 'reference (calibration)', tx)]
pd.DataFrame(rows).round(3)

copy input                 chrF  65.2 | SARI1  26.9 | copy 1.00 | formal 0.80 | short 0.00 | missing 0.05 | len 1.00
glossary only              chrF  68.8 | SARI1  49.5 | copy 0.45 | formal 0.58 | short 0.00 | missing 0.00 | len 0.94
glossary + lexicon         chrF  70.0 | SARI1  60.8 | copy 0.20 | formal 0.18 | short 0.00 | missing 0.00 | len 0.93
FINAL v6 (3-model vote | fluency w=10) chrF  73.0 | SARI1  71.0 | copy 0.04 | formal 0.02 | short 0.01 | missing 0.06 | len 0.87
reference (calibration)    chrF 100.0 | SARI1 100.0 | copy 0.00 | formal 0.06 | short 0.01 | missing 0.81 | len 0.82


,model,chrF,BLEU,SARI1,copy_rate,formal_left,too_short,missing,len_ratio
0,copy input,65.226,60.431,26.896,1.000,0.798,0.00,0.051,1.000
1,glossary only,68.769,66.271,49.527,0.455,0.576,0.00,0.000,0.942
2,glossary + lexicon,69.961,68.107,60.782,0.202,0.182,0.00,0.000,0.927
3,FINAL v6 (3-model vote | fluency w=10),72.985,73.583,70.967,0.040,0.020,0.01,0.061,0.875
4,reference (calibration),100.000,100.000,100.000,0.000,0.061,0.01,0.808,0.825


## 11. Error analysis
Where does the model still fail? (a) copies the input, (b) leaves royal words, (c) drops too much, (d) loses content words.

In [14]:
T = pd.DataFrame(dict(complex=tsrc, x=tx, reference=tref, output=final))
T['copied'] = T.output == T.complex
T['formal_left'] = T.output.apply(has_formal)
T['too_short'] = T.output.str.len() < 0.6*T.complex.str.len()
T['lost_words'] = [len(missing_words(o, x)) >= 2 for o, x in zip(T.output, T.x)]
print(T[['copied', 'formal_left', 'too_short', 'lost_words']].mean().round(3).to_string())
for col in ['too_short', 'lost_words', 'formal_left', 'copied']:
    sub = T[T[col]].head(3)
    print(f'\n##### {col}: {int(T[col].sum())} cases')
    for _, r in sub.iterrows():
        print('COMPLEX  :', r.complex); print('REFERENCE:', r.reference); print('OUTPUT   :', r.output); print('-'*50)

copied         0.04
formal_left    0.02
too_short      0.01
lost_words     0.00

##### too_short: 1 cases
COMPLEX  : ព្រះរាជា ទ្រង់ព្រះរាជានុញ្ញាតតាមសុំ ។
REFERENCE: ស្ដេចអនុញ្ញាតតាមសំណូមពរ ។
OUTPUT   : ស្ដេចបង្គាប់តាមសុំ ។
--------------------------------------------------

##### lost_words: 0 cases

##### formal_left: 2 cases
COMPLEX  : ពិធីបុណ្យរំលាយព្រះសពក៏បានចាត់ចែងឡើង ដោយរាជបរិវាររបស់ទ្រង់ មានទាំងស្ដេចក្រាញ់ ។
REFERENCE: គេរៀបពិធីបុណ្យរំលាយសព ដោយអ្នកបម្រើរបស់ស្ដេច ព្រមទាំងស្ដេចក្រាញ់ ។
OUTPUT   : ពិធីបុណ្យរំលាយសព ក៏ចាត់ចែងឡើង ដោយរាជបរិវាររបស់ស្ដេចមានទាំងស្ដេចក្រាញ់ ។
--------------------------------------------------
COMPLEX  : កូនដំរីនោះមានពណ៌សម្បុរ ព្រមទាំងរូបរាងល្អប្លែក អំពីដំរីធម្មតា អ្នករក្សាដំរីនាំដំណឹង ទៅក្រាបទូលព្រះចៅគ្រប់សព្វ ។
REFERENCE: កូនដំរីនោះមានពណ៌ និងរូបរាងល្អប្លែកពីដំរីធម្មតា អ្នកថែដំរីនាំដំណឹងទៅប្រាប់ស្ដេចទាំងអស់ ។
OUTPUT   : កូនដំរីនោះមានពណ៌សម្បុរ និងរូបរាងល្អជាងដំរីធម្មតា អ្នករក្សាដំរីនាំដំណឹងទៅប្រាប់ស្ដេចគ្រប់សព្វ ។
------------------------------------------

## 12. Try your own sentence
Use folk-tale style sentences (the model was trained on short sentences of 25–120 characters).

In [15]:
def simplify(text):
    x = normalize(text)
    return DECODE([x])[0]
print('--- 5 random unseen test sentences ---')
for i in random.sample(range(len(test)), 5):
    print('COMPLEX:', tsrc[i]); print('OUTPUT :', final[i]); print()
text = 'ព្រះរាជាទ្រង់ត្រាស់បង្គាប់ឲ្យមន្ត្រីចូលគាល់នៅមុខព្រះទីនាំង ។'   # replace with your own complex sentence
print('YOUR SENTENCE ->', simplify(text))

--- 5 random unseen test sentences ---
COMPLEX: ពួកសៀមជាន់នោះមានសេចក្ដីព្យាបាទ ធ្វើបាបខ្មែរអស្ចារ្យណាស់ ចាប់មនុស្សដោតស្លឹកត្រចៀកដឹកយកទៅដូចជាសត្វធាតុ ។
OUTPUT : ពួកសៀមជាន់នោះមានព្យាបាទ ធ្វើបាបខ្មែរអស្ចារ្យណាស់ ចាប់មនុស្សដោតស្លឹកត្រចៀកដឹកយកទៅដូចជាសត្វធាតុ ។

COMPLEX: រឿងនេះមិនបាត់ស្ងាត់ ជ្រាបដល់ព្រះចៅសៀម ទ្រង់ព្រះពិរោធយ៉ាងខ្លាំងពន់ពេក នឹកតូចព្រះទ័យក្រៃលែង ។
OUTPUT : រឿងនេះមិនបាត់ស្ងាត់ ជ្រាបស្ដេចសៀម ពិរោធខ្លាំងពេក នឹកតូចចិត្តក្រៃលែង ។

COMPLEX: កាលដែលស្រុកកើតចម្បាំងជាមួយនឹងសៀម, ព្រះរាជា ក៏ទ្រង់អំពាវនាវ ដល់អាណាប្រជានុរាស្ត្រគ្រប់ខេត្តខណ្ឌ ។
OUTPUT : ពេលស្រុកកើតចម្បាំងជាមួយសៀម ស្ដេចក៏អំពាវនាវដល់ប្រជាជនគ្រប់ខេត្តខណ្ឌ ។

COMPLEX: ទោះបីឪពុកខំនិយាយឃាត់យ៉ាងណា ក៏កុមារពុំយល់ស្របតាជាដាច់ខាត ។
OUTPUT : ក៏ឪពុកខំនិយាយឃាត់យ៉ាងណា ក៏កុមារមិនយល់ស្របតាជាដាច់ខាត ។

COMPLEX: ស្ដេចឲ្យប្រមូលព្រះរាជវង្សានុវង្សជួបជុំហើយ ទ្រង់ចាត់ការបិទទ្វារកំពែងវាំងស្រេច ទ្រង់ឲ្យអមាត្យបើកឆ្នុកបំពង់ពកនោះ ។
OUTPUT : ស្ដេចឲ្យប្រមូលញាតិវង្សជួបជុំ ហើយចាត់ការបិទទ្វារកំពែងវាំងស្រេច ឲ្យអមាត្យបើកឆ្នុកបំពង់ពកនោះ ។

YOUR SENTENCE -> ស្ដេច

In [16]:
# Interactive box (Colab). Type or paste any complex Khmer sentence and press Simplify.
try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    box = W.Textarea(value='ព្រះរាជបុត្រីទ្រង់ស្រង់ព្រះកាយនៅក្នុងស្រះព្រះរាជឧទ្យាន ហើយទ្រង់យាងត្រឡប់ចូលព្រះរាជវាំងវិញ ។', layout=W.Layout(width='95%', height='90px'))
    btn = W.Button(description='Simplify', button_style='success'); out_w = W.Output()
    def _click(_):
        with out_w:
            clear_output(); print('INPUT :', box.value.strip()); print('OUTPUT:', simplify(box.value.strip()))
    btn.on_click(_click); display(box, btn, out_w)
except Exception as e:
    print('Widget not available:', e)

Textarea(value='ព្រះរាជបុត្រីទ្រង់ស្រង់ព្រះកាយនៅក្នុងស្រះព្រះរាជឧទ្យាន ហើយទ្រង់យាងត្រឡប់ចូលព្រះរាជវាំងវិញ ។', …

Button(button_style='success', description='Simplify', style=ButtonStyle())

Output()

## 13. Summary
- **Data:** ~1,030 Complex→Simple pairs and the glossary were written and help by an AI assistant and are some of it is not verified by a native speaker yet.
- **Method:** glossary pre-pass → three fine-tuned mT5-small models (seed 42; seeds 43 and 44 with label smoothing) → guarded decoding with an extra fluency score from a pretrained XLM-RoBERTa (pseudo-log-likelihood), optional consensus voting across the three models, and an output clean-up (extra royal-word replacements, repeated-word removal). The extra glossary words were chosen from general royal vocabulary and checked on train/val only. The seed-42 model with guarded decoding only (no fluency score, no clean-up) was rated 70.0 % acceptable (n = 60, CI 57.5–80.1) by the Khmer team; report v6's rating next to it on the same 60 sentences. The model/decoding was chosen on the validation set; the test set was scored once. Lexicon rewriting of the input and edit-weighted training were also tried; they were worse than the seed-42 model, so they are not used.
- **Metrics:** report SARI-1, copy rate, formal-words-left, too-short and missing-words next to chrF/BLEU. Your headline accuracy should be the human **acceptable rate** from your team's rating with its 95% interval. chrF/BLEU against a single reference are inflated by copying and cannot see wrong words.
- **Known limits:** small training set; the glossary only covers ~55 royal words; long sentences may lose a clause; names and rare words are handled by the model only.
